[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sadirs/3ptWL-cov/blob/main/docs/examples/wlcovpy_covariance_colab.ipynb)

# Reproduce the covariance-matrix tutorial with `wlcovpy` from PyPI

This Colab notebook reproduces the compact workflow from the [3ptWL-cov covariance-matrix tutorial](https://3ptwl-cov.readthedocs.io/en/latest/tutorials/covariance-matrix.html) using the published **`wlcovpy==1.0.1`** package.

It is self-contained: it installs the native build requirements, downloads the version-matched `C_ell` and angular-grid fixtures, defines the small NumPy helper layer used by the repository tutorial, computes the documented `6 x 6` covariance matrix, validates the result, and plots it.

> Run the cells in order in a fresh Colab runtime. The first installation may take a minute because the C/Cython extension is compiled locally.


In [ ]:
import subprocess
import sys

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        [
            "apt-get", "install", "-y", "-qq",
            "build-essential", "libgsl-dev", "python3-dev",
        ],
        check=True,
    )

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-cache-dir", "--quiet",
        "wlcovpy==1.0.1", "matplotlib>=3.5",
    ],
    check=True,
)

print("Native dependencies and wlcovpy 1.0.1 are installed.")


## 1. Import the package and download the tutorial inputs

The input files come from the matching `v1.0.1` GitHub release tag, so the notebook does not depend on a local clone of the repository.


In [ ]:
from importlib.metadata import version
from pathlib import Path
from urllib.request import urlretrieve
import gc
import os

import matplotlib.pyplot as plt
import numpy as np
from wlcovpy import wlcov

installed_version = version("wlcovpy")
assert installed_version == "1.0.1", installed_version

WORKDIR = Path("/content/wlcovpy_covariance") if IN_COLAB else Path.cwd() / "wlcovpy_covariance"
WORKDIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)

release_base = "https://raw.githubusercontent.com/sadirs/3ptWL-cov/v1.0.1/tests/input"
CLS_FILE = WORKDIR / "Cls_ep2.txt"
THETA_FILE = WORKDIR / "theta_array.txt"

for filename, destination in [
    ("Cls_ep2.txt", CLS_FILE),
    ("theta_array.txt", THETA_FILE),
]:
    urlretrieve(f"{release_base}/{filename}", destination)

ell, cls = np.loadtxt(CLS_FILE, unpack=True)
theta_all = np.loadtxt(THETA_FILE)

assert ell.shape == cls.shape and ell.size > 0
assert theta_all.ndim == 1 and theta_all.size >= 4

print(f"wlcovpy version: {installed_version}")
print(f"Downloaded {ell.size} spectrum rows and {theta_all.size} angular bins")
print(f"Working directory: {WORKDIR}")


## 2. Define the covariance helpers

These functions mirror `tests/python/covariance_example.py` from the tutorial. Each selected pair of angular bins becomes one covariance-matrix coordinate; only one triangle is evaluated, then symmetry fills the other half.


In [ ]:
def add_noise_to_column(input_file, output_file, noise):
    # Add a constant noise term to the C_ell column.
    with Path(input_file).open() as infile, Path(output_file).open("w") as outfile:
        for line in infile:
            parts = line.split()
            if len(parts) != 2:
                outfile.write(line)
                continue
            try:
                outfile.write(f"{parts[0]}\t{float(parts[1]) + noise:.17g}\n")
            except ValueError:
                outfile.write(line)


def build_mask(dim, rows, diagonals, symm=True):
    # Select angular-bin pairs after row and diagonal exclusions.
    mask = np.ones((dim, dim), dtype=bool)
    mask[:rows, :] = False
    mask[:, :rows] = False

    offsets = np.abs(np.subtract.outer(np.arange(dim), np.arange(dim)))
    for diagonal in range(diagonals):
        mask[offsets == diagonal] = False

    return np.triu(mask) if symm else mask


def get_valid_indices(mask):
    rows, cols = np.where(mask)
    return np.column_stack((rows, cols))


def calculate_integral(theta1, theta2, thetap1, thetap2, m, mp, ppp, inputfile):
    # Run one native wlcov covariance integral and release its memory.
    model = wlcov(default=False)
    model.set(
        {
            "theta1": float(theta1),
            "theta2": float(theta2),
            "thetap1": float(thetap1),
            "thetap2": float(thetap2),
            "clsfile": str(inputfile),
            "m": int(m),
            "mp": int(mp),
            "ppp": int(ppp),
            "options": "",
            "verbose": 0,
            "verbose_log": 0,
        }
    )
    try:
        model.Run()
        return float(model.getIntegral())
    finally:
        model.clean_all()


def compute_cov_noise(
    *, rows, diagonals, dim, m, mp, ppp, noise,
    theta, inputfile, outputfile=None,
):
    # Build the symmetric covariance matrix for the selected angular pairs.
    theta = np.asarray(theta, dtype=float)
    if theta.ndim != 1 or theta.size != dim:
        raise ValueError(f"theta must be a one-dimensional array of length {dim}")

    noisy_cls = WORKDIR / "Cls_temp.txt"
    add_noise_to_column(inputfile, noisy_cls, noise)

    indices = get_valid_indices(build_mask(dim, rows, diagonals))
    covariance = np.zeros((len(indices), len(indices)), dtype=float)

    try:
        for i, first_pair in enumerate(indices):
            if i % 5 == 0:
                print(f"Progress: {i}/{len(indices)}")

            theta1, theta2 = theta[first_pair]
            for j in range(i + 1):
                thetap1, thetap2 = theta[indices[j]]
                value = calculate_integral(
                    theta1, theta2, thetap1, thetap2,
                    m, mp, ppp, noisy_cls,
                )
                covariance[i, j] = value
                covariance[j, i] = value
    finally:
        noisy_cls.unlink(missing_ok=True)
        gc.collect()

    if outputfile is not None:
        np.savetxt(outputfile, covariance)
        print(f"Saved covariance matrix to {outputfile}")

    return covariance, indices


## 3. Inspect the inputs and selected angular pairs

As in the documentation's compact example, use the first four angular values and exclude the main diagonal. This leaves six unique angular-bin pairs.


In [ ]:
theta = theta_all[:4]
rows = 0
diagonals = 1
mask = build_mask(dim=len(theta), rows=rows, diagonals=diagonals)
selected_pairs = get_valid_indices(mask)

assert selected_pairs.shape == (6, 2)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

axes[0].loglog(ell, cls, color="#1f77b4", lw=2)
axes[0].set_title(r"Input $C_\ell$ table")
axes[0].set_xlabel(r"$\ell$")
axes[0].set_ylabel(r"$C_\ell$")
axes[0].grid(True, which="both", alpha=0.25)

im = axes[1].imshow(mask, cmap="Greens", interpolation="nearest")
axes[1].set_title("Selected covariance entries")
axes[1].set_xlabel("theta index")
axes[1].set_ylabel("theta index")
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04, label="selected")

inputs_plot = WORKDIR / "inputs_and_mask.png"
fig.savefig(inputs_plot, dpi=180, bbox_inches="tight")
plt.show()

print("theta =", theta)
print("selected index pairs =", selected_pairs.tolist())


## 4. Compute the compact covariance matrix

The low `ppp=4` setting matches the quick documentation workflow. It is intended for reproducibility and smoke testing, not convergence-grade science production.


In [ ]:
settings = dict(
    rows=rows,
    diagonals=diagonals,
    dim=len(theta),
    m=0,
    mp=0,
    ppp=4,
    noise=6.1e-11,
)

COVARIANCE_FILE = WORKDIR / "analytic_covariance_example.txt"
covariance, covariance_pairs = compute_cov_noise(
    theta=theta,
    inputfile=CLS_FILE,
    outputfile=COVARIANCE_FILE,
    **settings,
)

assert covariance.shape == (6, 6)
assert np.isfinite(covariance).all()
assert np.allclose(covariance, covariance.T, rtol=0.0, atol=0.0)
assert COVARIANCE_FILE.exists()

print(f"Covariance shape: {covariance.shape}")
print(f"Finite values:    {np.isfinite(covariance).all()}")
print(f"Exactly symmetric: {np.array_equal(covariance, covariance.T)}")
covariance


## 5. Plot and save the result

The left panel shows `log10(abs(covariance))`; the right panel shows the magnitudes of the upper-triangle entries, matching the tutorial presentation.


In [ ]:
abs_cov = np.abs(covariance)
log_cov = np.full_like(abs_cov, np.nan, dtype=float)
positive = abs_cov > 0
log_cov[positive] = np.log10(abs_cov[positive])

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

im = axes[0].imshow(log_cov, cmap="viridis", interpolation="nearest")
axes[0].set_title(r"$\log_{10}|\mathrm{covariance}|$")
axes[0].set_xlabel("matrix index")
axes[0].set_ylabel("matrix index")
fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

upper_values = abs_cov[np.triu_indices_from(abs_cov)]
upper_values = upper_values[upper_values > 0]
axes[1].semilogy(
    np.arange(1, len(upper_values) + 1),
    upper_values,
    "o-",
    color="#d62728",
    lw=1.6,
)
axes[1].set_title("Upper-triangle magnitudes")
axes[1].set_xlabel("entry number")
axes[1].set_ylabel("absolute covariance")
axes[1].grid(True, which="both", alpha=0.25)

covariance_plot = WORKDIR / "covariance_summary.png"
fig.savefig(covariance_plot, dpi=180, bbox_inches="tight")
plt.show()

print(f"PASS: reproduced the tutorial's finite {covariance.shape} covariance matrix.")
print(f"Matrix: {COVARIANCE_FILE}")
print(f"Plot:   {covariance_plot}")


## 6. Optional full R2D2 paper-data configuration

The tutorial also records the larger production-reference settings below. They require many more native integrations, so this cell is disabled by default. Set `RUN_FULL_R2D2 = True` only when you intentionally want the longer run.


In [ ]:
RUN_FULL_R2D2 = False

if RUN_FULL_R2D2:
    full_settings = dict(
        rows=7,
        diagonals=4,
        dim=20,
        m=2,
        mp=2,
        ppp=20,
        noise=6.1e-11,
    )
    full_covariance, full_pairs = compute_cov_noise(
        theta=theta_all,
        inputfile=CLS_FILE,
        outputfile=WORKDIR / "analytic_covariance_r2d2.txt",
        **full_settings,
    )
    print("Full covariance shape:", full_covariance.shape)
else:
    print("Full R2D2 run skipped. Set RUN_FULL_R2D2 = True to enable it.")


## Troubleshooting

- Run all cells from the top in a fresh runtime; the compiled extension must be installed before importing `wlcovpy`.
- If an interrupted installation leaves the runtime in an inconsistent state, use **Runtime -> Restart session**, then run all cells again.
- The notebook intentionally pins both the package and fixture URLs to `v1.0.1` so future releases do not silently change this reproduction.
